# Part C: Compute evaluation performances: Garonne

This notebook contains all the code used to compute the evaluation performances (space and space-time validation) of the models calibrated using the three geology maps to infer their respective HRUs. 


Alongside the other notebooks, it covers all the analysis performed in: "Assessing the Impact of Geological Map Detail on Process-Based and Data-Driven Hydrological Models" paper by do Nascimento et al. (in review). To be able to run this notebook, please ensure that you have downloaded the acompanying data of the paper. All links can be found in the data section of the paper.

Author: Thiago Nascimento (thiago.nascimento@eawag.ch)

# Import the modules

In [ ]:
import pandas as pd
import datetime as datetime
import matplotlib.pyplot as plt
import numpy as np
import spotpy
import time
import os
import tqdm as tqdm
import hydroanalysis
from utils.functions import find_max_unique_rows
from utils.functions import find_iterative_immediate_downstream
import geopandas as gpd
import re

#warnings.filterwarnings("ignore")

## Set the path to the data

In [2]:
# Path to where the EStreams dataset is stored
# Eawag
path_estreams = r'/Users/nascimth/Documents/data/EStreams'

## Mac
#path_estreams = r'/Users/thiagomedeirosdonascimento/Downloads/Python 2/Scripts/estreams_part_b/data/EStreams'

path_data = r"/Users/nascimth/Documents/data"

## Read the files

In [3]:
# Read the dataset network
network_estreams = pd.read_csv(path_estreams+'/streamflow_gauges/estreams_gauging_stations.csv', encoding='utf-8')
network_estreams.set_index("basin_id", inplace = True)

# Convert 'date_column' and 'time_column' to datetime
network_estreams['start_date'] = pd.to_datetime(network_estreams['start_date'])
network_estreams['end_date'] = pd.to_datetime(network_estreams['end_date'])

# Convert to list both the nested_catchments and the duplicated_suspect columns
network_estreams['nested_catchments'] = network_estreams['nested_catchments'].apply(lambda x: x.strip("[]").replace("'", "").split(", "))

# Remove the brackets and handle NaN values
network_estreams['duplicated_suspect'] = network_estreams['duplicated_suspect'].apply(
    lambda x: x.strip("[]").replace("'", "").split(", ") if isinstance(x, str) else x)

# Set the nested catchments as a dataframe
nested_catchments = pd.DataFrame(network_estreams['nested_catchments'])

# Now we add the outlet to the list (IF it was not before):
# Ensure that the basin_id is in the nested_catchments
for basin_id in nested_catchments.index:
    if basin_id not in nested_catchments.at[basin_id, 'nested_catchments']:
        nested_catchments.at[basin_id, 'nested_catchments'].append(basin_id)



# Attributes already filtered previously:
#estreams_attributes = pd.read_csv('data/exploration/estreams_attributes_filtered_moselle_sm_su_tog.csv', encoding='utf-8')
estreams_attributes = pd.read_csv('../data/estreams_attributes_filtered_quality_geology_v01.csv', encoding='utf-8')

estreams_attributes.set_index("basin_id", inplace = True)

# Convert to list both the nested_catchments and the duplicated_suspect columns
estreams_attributes['nested_catchments'] = estreams_attributes['nested_catchments'].apply(lambda x: x.strip("[]").replace("'", "").split(", "))

# Remove the brackets and handle NaN values
estreams_attributes['duplicated_suspect'] = estreams_attributes['duplicated_suspect'].apply(
    lambda x: x.strip("[]").replace("'", "").split(", ") if isinstance(x, str) else x)

estreams_attributes.sort_index(inplace = True) 

In [4]:
# Geological attributes (regional scale)
geology_regional_31_classes_moselle = pd.read_csv("../data/estreams_geology_moselle_regional_attributes.csv", encoding='utf-8')

geology_regional_31_classes_moselle.set_index("basin_id", inplace = True)

# Create a dictionary to map permeability classes to corresponding columns
permeability_columns = {
    "high": ["lit_fra_Alluvium", 'lit_fra_Coal', 'lit_fra_Conglomerate', 'lit_fra_Gravel and sand',
             'lit_fra_Sand', 'lit_fra_Sand and gravel', 'lit_fra_Sandstone and conglomerate', 'lit_fra_Sandstone'
        ],
    
    "medium": ['lit_fra_Limestone', 'lit_fra_Sandstone and marl', 'lit_fra_Sandstone and schist',
              'lit_fra_Sandstone, conglomerate and marl',

              'lit_fra_Arkose', 'lit_fra_Dolomite rock', 'lit_fra_Limestone and marl', 'lit_fra_Marl', 
             'lit_fra_Marl and dolomite', 'lit_fra_Marl and limestone', 'lit_fra_Marl and sandstone',
               'lit_fra_Sandstone and siltstone', 'lit_fra_Sandstone, siltstone and schist', 
              'lit_fra_Schist and sandstone', 'lit_fra_Silt',  'lit_fra_Silt and schist', 'lit_fra_Siltstone, sandstone and schist'
              
             ],
    
    "low": ['lit_fra_Cristallin basement', 'lit_fra_Plutonic rock',  'lit_fra_Quarzite',
                    'lit_fra_Schist','lit_fra_Volcanic rock' 
                   ]
}

# Iterate over the permeability columns and calculate the area for each class
for permeability_class, columns in permeability_columns.items():
    geology_regional_31_classes_moselle[f'area_perm_{permeability_class}'] = geology_regional_31_classes_moselle[columns].sum(axis=1)

# Drop unnecessary columns
geology_regional_31_classes_moselle = geology_regional_31_classes_moselle[["area_perm_high", "area_perm_medium", "area_perm_low"]]

# Rename the columns
geology_regional_31_classes_moselle.columns = ["perm_high_regi", "perm_medium_regi", "perm_low_regi"]

# Display the updated DataFrame
geology_regional_31_classes_moselle

geology_regional_31_classes_moselle["baseflow_index"] = estreams_attributes["baseflow_index"]
geology_regional_31_classes_moselle.corr(method="pearson")

# Concatenation
estreams_attributes[["perm_high_regi", "perm_medium_regi", "perm_low_regi"]] = geology_regional_31_classes_moselle[["perm_high_regi", "perm_medium_regi", "perm_low_regi"]]

# Adjust the three categories for also global dataset
estreams_attributes["perm_high_glob2"] = estreams_attributes["perm_high_glob"]
estreams_attributes["perm_medium_glob2"] = estreams_attributes["perm_medium_glob"] + estreams_attributes["perm_low_glob"]
estreams_attributes["perm_low_glob2"] = estreams_attributes["perm_verylow_glob"]

###########################################################################################################################
# Adjust the columns of the dataset:
for basin_id in estreams_attributes.index.tolist():

    # Extract and divide by 100
    v1 = estreams_attributes.loc[basin_id, "perm_high_regi"] / 100
    v2 = estreams_attributes.loc[basin_id, "perm_medium_regi"] / 100
    v3 = estreams_attributes.loc[basin_id, "perm_low_regi"] / 100

    # Round all values to one decimal place
    v1 = round(v1, 2)
    v2 = round(v2, 2)
    v3 = round(v3, 2)

    # Ensure the sum is exactly 1 by adjusting the largest value
    diff = 1 - (v1 + v2 + v3)

    if diff != 0:
        # Adjust the value that was the largest before rounding
        if max(v1, v2, v3) == v1:
            v1 += diff
        elif max(v1, v2, v3) == v2:
            v2 += diff
        else:
            v3 += diff

    # Assign back
    estreams_attributes.loc[basin_id, "perm_high_regi"] = v1 * 100
    estreams_attributes.loc[basin_id, "perm_medium_regi"] = v2 * 100
    estreams_attributes.loc[basin_id, "perm_low_regi"] = v3 * 100


for basin_id in estreams_attributes.index.tolist():

    # Extract and divide by 100
    v1 = estreams_attributes.loc[basin_id, "perm_high_glob2"] / 100
    v2 = estreams_attributes.loc[basin_id, "perm_medium_glob2"] / 100
    v3 = estreams_attributes.loc[basin_id, "perm_low_glob2"] / 100

    # Round all values to one decimal place
    v1 = round(v1, 2)
    v2 = round(v2, 2)
    v3 = round(v3, 2)

    # Ensure the sum is exactly 1 by adjusting the largest value
    diff = 1 - (v1 + v2 + v3)

    if diff != 0:
        # Adjust the value that was the largest before rounding
        if max(v1, v2, v3) == v1:
            v1 += diff
        elif max(v1, v2, v3) == v2:
            v2 += diff
        else:
            v3 += diff

    # Assign back
    estreams_attributes.loc[basin_id, "perm_high_glob2"] = v1 * 100
    estreams_attributes.loc[basin_id, "perm_medium_glob2"] = v2 * 100
    estreams_attributes.loc[basin_id, "perm_low_glob2"] = v3 * 100

In [5]:
# Define the functions
def obj_fun_nsee(observations, simulation, expo=0.5):
    """
    Calculate the Normalized Squared Error Efficiency (NSEE) while ensuring that
    NaNs in simulation are NOT masked (only NaNs in observations are masked).

    Parameters:
        observations (array-like): Observed values (with fixed NaNs).
        simulation (array-like): Simulated values (can contain NaNs).
        expo (float, optional): Exponent applied to observations and simulations. Default is 1.0.

    Returns:
        float: NSEE score (higher values indicate worse performance).
    """
    observations = np.asarray(observations)
    simulation = np.asarray(simulation)

    # Mask only NaNs in observations
    mask = ~np.isnan(observations)
    obs = observations[mask]
    sim = simulation[mask]  # Keep all simulated values, even NaNs

    # If simulation contains NaNs after masking observations, return penalty
    if np.isnan(sim).any():
        return 10.0  # Large penalty if NaNs appear in the simulation

    metric = np.sum((sim**expo - obs**expo)**2) / np.sum((obs**expo - np.mean(obs**expo))**2)
    
    return float(metric)


def obj_fun_kge(observations, simulation):
    """
    Calculate the KGE-2012 objective function, ensuring that NaNs in simulation are NOT masked.
    
    Parameters:
        observations (array-like): Observed values (with fixed NaNs).
        simulation (array-like): Simulated values (can contain NaNs).

    Returns:
        float: KGE-2012 score (higher values indicate worse performance).
    """
    observations = np.asarray(observations)
    simulation = np.asarray(simulation)

    # Mask only NaNs in observations
    mask = ~np.isnan(observations)
    obs = observations[mask]
    sim = simulation[mask]  # Keep all simulated values, even NaNs

    # Check if there are NaNs in the simulation after masking obs
    if np.isnan(sim).any():
        return 10.0  # Large penalty if the simulation contains NaNs
    
    obs_mean = np.mean(obs)
    sim_mean = np.mean(sim)

    r = np.corrcoef(obs, sim)[0, 1]
    alpha = np.std(sim) / np.std(obs)
    beta = sim_mean / obs_mean

    kge = np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)  # KGE-2012

    return float(kge)

In [6]:
# First we define the outlet of the Moselle to be used
outlets = ["FR001604"]
nested_cats_df = nested_catchments.loc[outlets, :]

# Now we save our dataframes in a dictionary of dataframes. One dataframe for each watershed. 

nested_cats_filtered = find_max_unique_rows(nested_cats_df)                                  # Filter only the catchemnts using the function stated before
nested_cats_filtered_df = nested_catchments.loc[nested_cats_filtered, :]                     # Here we filter the catchemnts for the list (again, after we apply our function):

# Store the variables for the selected catchments in a list of dataframes now for only the ones above 20 cats:
estreams_attributes_dfs = {}
for catchment in tqdm.tqdm(nested_cats_filtered):
    # Retrieve the nested list of catchments for the current catchment
    nested_clip = nested_cats_filtered_df.loc[catchment, 'nested_catchments']
    
    # Filter values to include only those that exist in the index of estreams_attributes
    nested_clip = [value for value in nested_clip if value in estreams_attributes.index]
    
    # Filter the estreams_attributes DataFrame based on the filtered nested_clip
    cat_clip = estreams_attributes.loc[nested_clip, :]
    
    # Store the resulting DataFrame in the dictionary
    estreams_attributes_dfs[catchment] = cat_clip

# Here we can save the length of each watershed (number of nested catchemnts)
catchment_lens = pd.DataFrame(index = estreams_attributes_dfs.keys())
for catchment, data in estreams_attributes_dfs.items():
    catchment_lens.loc[catchment, "len"] = len(data)

# Now we can filter it properly:
nested_cats_filtered_abovevalue = catchment_lens[catchment_lens.len >= 10]

# # Here we filter the catchemnts for the list (again, after we apply our function):
nested_cats_filtered_abovevalue_df = nested_catchments.loc[nested_cats_filtered_abovevalue.index, :]

# Store the variables for the selected catchments in a list of dataframes now for only the ones above 20 cats:
estreams_attributes_dfs = {}

for catchment in tqdm.tqdm(nested_cats_filtered_abovevalue_df.index):
    # Retrieve the nested list of catchments for the current catchment
    nested_clip = nested_cats_filtered_abovevalue_df.loc[catchment, 'nested_catchments']
    
    # Filter values to include only those that exist in the index of estreams_attributes
    nested_clip = [value for value in nested_clip if value in estreams_attributes.index]
    
    # Filter the estreams_attributes DataFrame based on the filtered nested_clip
    cat_clip = estreams_attributes.loc[nested_clip, :]
    
    # Store the resulting DataFrame in the dictionary
    estreams_attributes_dfs[catchment] = cat_clip

# Adjust and clip it:
estreams_attributes_clipped = estreams_attributes_dfs["FR001604"]

# Convert 'date_column' and 'time_column' to datetime
estreams_attributes_clipped['start_date'] = pd.to_datetime(estreams_attributes_clipped['start_date'])
estreams_attributes_clipped['end_date'] = pd.to_datetime(estreams_attributes_clipped['end_date'])


#estreams_attributes_clipped_filters = estreams_attributes_clipped[estreams_attributes_clipped.end_date >= "2010"]
#estreams_attributes_clipped_filters = estreams_attributes_clipped_filters[estreams_attributes_clipped_filters.start_date <= "2002"]

# Here we retrieve the conectivity (from EStreams computation)
# Load the nested catchments CSV file
df = pd.read_excel("../data/nested_catchments.xlsx")

# Rename columns for clarity
df = df.rename(columns={df.columns[1]: "basin_id", df.columns[2]: "connected_basin_id"})
df = df.drop(columns=[df.columns[0]])  # Drop the unnamed index column

100%|██████████| 1/1 [00:00<00:00, 803.35it/s]


In [7]:
# Read the dataset network
estreams_attributes_clipped_filters = pd.read_csv(R'../data/network_estreams_garonne_22_gauges.csv', encoding='utf-8')
estreams_attributes_clipped_filters.set_index("basin_id", inplace = True)
estreams_attributes_clipped_filters

,Unnamed: 0,gauge_id,gauge_name,gauge_country,gauge_provider,river,lon_snap,lat_snap,lon,lat,...,perm_medium_cont2,perm_low_cont2,lakes_depth_mean,lakes_depth_mean_norm,q_mean_yr,p_mean_yr,norm_ups_cap,NSE,NSElstm,group
basin_id,,,,,,,,,,,,,,,,,,,,,
FR004136,0,O149431003,Le Touyre Ã Lavelanet et Ã Saint-Quentin-la-...,FR,FR_EAUFRANCE,Le Touyre à Lavelanet et à Saint-Quentin-la-Tour,1.850824,42.940862,1.850824,42.940862,...,52.863,47.137,NaN,0.000000,846.800,888.775,0.000000,0.700021,0.815218,Group_1
FR001547,1,O062401001,O0624010,FR,FR_EAUFRANCE,Le Volp à Montberaud et à Sainte-Croix-Volvestre,1.141456,43.144783,1.141456,43.144783,...,100.000,0.000,NaN,0.000000,337.990,819.790,0.000000,0.823456,0.876788,Group_1
FR004091,2,O023402001,Le Ger Ã Aspet [2],FR,FR_EAUFRANCE,Le Ger à Aspet [2],0.795350,43.021419,0.795350,43.021419,...,49.849,50.151,NaN,0.000000,919.800,1040.250,0.000000,0.723698,0.848994,Group_1
FR001592,3,O221501001,O2215010,FR,FR_EAUFRANCE,La Saune à Quint-Fonsegrives,1.548308,43.574698,1.548308,43.574698,...,97.803,0.000,3.648649,12.385889,117.165,657.730,0.018831,0.809932,0.811849,Group_1
FR001577,4,O158461001,O1584610,FR,FR_EAUFRANCE,Le Douctouyre à Dun et à Vira [Engraviès],1.772643,43.043560,1.772643,43.043560,...,94.518,5.483,NaN,0.000000,380.330,832.930,0.000000,0.852887,0.849163,Group_1
FR004140,5,O163401001,La VixiÃ¨ge Ã Belpech,FR,FR_EAUFRANCE,La Vixiège à Belpech,1.749951,43.201718,1.749951,43.201718,...,92.188,0.000,NaN,0.000000,195.640,718.685,0.000000,0.822625,0.817920,Group_1
FR004149,6,O184402001,La LÃ¨ze Ã LÃ©zat-sur-LÃ¨ze,FR,FR_EAUFRANCE,La Lèze à Lézat-sur-Lèze,1.352878,43.280809,1.352878,43.280809,...,92.809,0.664,7.571429,12.564601,178.485,775.260,0.016207,0.853714,0.893136,Group_1
FR004152,7,O187401001,La LÃ¨ze Ã Labarthe-sur-LÃ¨ze,FR,FR_EAUFRANCE,La Lèze à Labarthe-sur-Lèze,1.406594,43.451190,1.406594,43.451190,...,81.174,0.456,7.571429,8.638464,161.330,740.220,0.011670,0.866051,0.889248,Group_1
FR001593,8,O222251001,O2222510,FR,FR_EAUFRANCE,L'Hers à Toulouse [Pont de Périole],1.480100,43.627770,1.480100,43.627770,...,81.003,0.000,4.817610,9.651561,138.335,665.030,0.014513,0.820100,0.826636,Group_1


In [8]:
# Python implementation
from superflexpy.framework.unit import Unit
from superflexpy.framework.node import Node
from superflexpy.framework.network import Network

from superflexpy.implementation.elements.hbv import UnsaturatedReservoir, PowerReservoir

from superflexpy.implementation.numerical_approximators.implicit_euler import ImplicitEulerPython
from superflexpy.implementation.root_finders.pegasus import PegasusPython

# Numba implementation:
from superflexpy.implementation.root_finders.pegasus import PegasusNumba
from superflexpy.implementation.numerical_approximators.implicit_euler import ImplicitEulerNumba

from superflexpy.implementation.elements.hbv import PowerReservoir
from superflexpy.framework.unit import Unit
from superflexpy.implementation.elements.thur_model_hess import SnowReservoir, UnsaturatedReservoir, PowerReservoir, HalfTriangularLag

from superflexpy.implementation.elements.structure_elements import Transparent, Junction, Splitter
from superflexpy.framework.element import ParameterizedElement

In [9]:
#root_finder = PegasusNumba()
#num_app = ImplicitEulerNumba(root_finder=root_finder)


# Root finder / approximation
class PegasusNumbaSafe(PegasusNumba):
    def __init__(self):
        super().__init__()
        self._iter_max = 100  # allow up to 50 iterations
        self._tol = 1e-6     # stricter tolerance


root_finder = PegasusNumbaSafe()
num_app = ImplicitEulerNumba(root_finder)

class ParameterizedSingleFluxSplitter(ParameterizedElement):
    _num_downstream = 2
    _num_upstream = 1
    
    def set_input(self, input):

        self.input = {'Q_in': input[0]}

    def get_output(self, solve=True):

        split_par = self._parameters[self._prefix_parameters + 'splitpar']

        output1 = [self.input['Q_in'] * split_par]
        output2 = [self.input['Q_in'] * (1 - split_par)]
        
        return [output1, output2]   
    
    
lower_splitter = ParameterizedSingleFluxSplitter(
    parameters={'splitpar': 0.5},
    id='lowersplitter'
)

lower_splitter_medium = ParameterizedSingleFluxSplitter(
    parameters={'splitpar': 0.6},
    id='lowersplitter'
)

lower_splitter_high = ParameterizedSingleFluxSplitter(
    parameters={'splitpar': 0.7},
    id='lowersplitter'
)

# Fluxes in the order P, T, PET
upper_splitter = Splitter(
    direction=[
        [0, 1, None],    # P and T go to the snow reservoir
        [2, None, None]  # PET goes to the transparent element
    ],
    weight=[
        [1.0, 1.0, 0.0],
        [0.0, 0.0, 1.0]
    ],
    id='upper-splitter'
)

snow = SnowReservoir(
    parameters={'t0': 0.0, 'k': 0.01, 'm': 2.0},
    states={'S0': 0.0},
    approximation=num_app,
    id='snow'
)

upper_transparent = Transparent(
    id='upper-transparent'
)

upper_junction = Junction(
    direction=[
        [0, None],
        [None, 0]
    ],
    id='upper-junction'
)


unsaturated = UnsaturatedReservoir(
    parameters={'Smax': 150.0, 'Ce': 1.0, 'm': 0.01, 'beta': 2.0},
    states={'S0': 10.0},
    approximation=num_app,
    id='unsaturated'
)

fast = PowerReservoir(
    parameters={'k': 0.01, 'alpha': 2.0},
    states={'S0': 0.0},
    approximation=num_app,
    id='fast'
)

slow = PowerReservoir(
    parameters={'k': 1e-4, 'alpha': 1.0},
    states={'S0': 0.0},
    approximation=num_app,
    id='slow'
)

slowhigh = PowerReservoir(
    parameters={'k': 1e-4, 'alpha': 2.0},
    states={'S0': 0.0},
    approximation=num_app,
    id='slowhigh'
)


lower_junction = Junction(
    direction=[
        [0, 0]
    ],
    id='lower-junction'
)

lag_fun = HalfTriangularLag(
    parameters={'lag-time': 4.0},
    states={'lag': None},
    id='lag-fun'
)

lower_transparent = Transparent(
    id='lower-transparent'
)

lower_transparent2 = Transparent(
    id='lower-transparent2'
)

general = Unit(
    layers=[
        [upper_splitter],
        [snow, upper_transparent],
        [upper_junction],
        [unsaturated],
        [lower_splitter],
        [slow, lag_fun],
        [lower_transparent, fast],
        [lower_junction],
    ],
    id='general'
)

low = Unit(
    layers=[
        [upper_splitter],
        [snow, upper_transparent],
        [upper_junction],
        [unsaturated],
        [fast],
    ],
    id='low'
)

high = Unit(
    layers=[
        [upper_splitter],
        [snow, upper_transparent],
        [upper_junction],
        [unsaturated],
        [slowhigh],
    ],
    id='high'
)

In [10]:
import os
import glob

# Dictionary to store all parameter dicts
all_param_dicts = {}

# Loop through all CSVs in the current directory
for filepath in glob.glob("../results/groups/*garonne*comp*.csv"):
    file_key = os.path.splitext(os.path.basename(filepath))[0]  # Strip .csv
    
    param_dict = {}

    # Read file and parse lines
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith(","):  # Skip empty or malformed lines
                continue
            parts = line.split(",")
            if len(parts) == 2:
                key, value = parts
                try:
                    param_dict[key] = float(value)
                except ValueError:
                    pass  # Skip lines where value is not a float
            else:
                pass  # Skip malformed lines

    # Store the parsed dictionary
    all_param_dicts[file_key] = param_dict


In [11]:
catchments_ids = estreams_attributes_clipped_filters.index.tolist()

def calculate_hydro_year(date, first_month=10):
    """
    This function calculates the hydrological year from a date. The
    hydrological year starts on the month defined by the parameter first_month.

    Parameters
    ----------
    date : pandas.core.indexes.datetimes.DatetimeIndex
        Date series
    first_month : int
        Number of the first month of the hydrological year

    Returns
    -------
    numpy.ndarray
        Hydrological year time series
    """

    hydrological_year = date.year.values.copy()
    hydrological_year[date.month >= first_month] += 1

    return hydrological_year

def run_model_superflexpy(catchments_ids, best_params_dict_model, perm_areas_model):
    # Run the iterative function
    iterative_immediate_downstream = find_iterative_immediate_downstream(df, catchments_ids)

    # Convert results to a DataFrame for display
    iterative_downstream_df = pd.DataFrame(iterative_immediate_downstream.items(), 
                                        columns=['basin_id', 'immediate_downstream_basin'])


    # Assuming the DataFrame has columns 'basin_id' and 'downstream_id'
    topology_list = {basin: None for basin in catchments_ids}  # Default to None

    # Filter DataFrame for relevant basin_ids and update topology
    for _, row in iterative_downstream_df.iterrows():
        if row['basin_id'] in topology_list:
            topology_list[row['basin_id']] = row['immediate_downstream_basin']

    # Generate Nodes dynamically and assign them as global variables
    catchments = [] # Dictionary to store nodes
    
    general = Unit(
    layers=[
        [upper_splitter],
        [snow, upper_transparent],
        [upper_junction],
        [unsaturated],
        [lower_splitter],
        [slow, lag_fun],
        [lower_transparent, fast],
        [lower_junction],
    ],
    id='general')

    low = Unit(
        layers=[
        [upper_splitter],
        [snow, upper_transparent],
        [upper_junction],
        [unsaturated],
        [lower_splitter],
        [slow, lag_fun],
        [lower_transparent, fast],
        [lower_junction],
    ],
        id='low')

    high = Unit(
        layers=[
        [upper_splitter],
        [snow, upper_transparent],
        [upper_junction],
        [unsaturated],
        [lower_splitter],
        [slow, lag_fun],
        [lower_transparent, fast],
        [lower_junction],
    ],
        id='high')

    for cat_id in catchments_ids:
        node = Node(
            units=[high, general, low],  # Use unit from dictionary or default
            weights=perm_areas_model[cat_id],
            area=areas.get(cat_id),  # Use predefined area or default
            id=cat_id
        )
        catchments.append(node)  # Store in the list

        # Assign the node as a global variable
        globals()[cat_id] = node


    # Ensure topology only includes nodes that exist in `catchments_ids`
    topology = {
        cat_id: upstream if upstream in catchments_ids else None
        for cat_id, upstream in topology_list.items() if cat_id in catchments_ids
    }

    # Create the Network
    model = Network(
        nodes=catchments,  # Pass list of Node objects
        topology=topology  
    )

    model.reset_states()

    # Set inputs for each node using the manually defined dictionary
    for cat in catchments:
        cat.set_input(inputs[cat.id])  # Correct way to set inputs

    model.set_timestep(1.0)
    model.set_parameters(best_params_dict_model)
    
    print(model.get_parameters())
    print(model._content[1].get_parameters())
    print(model._content[-1].get_parameters())

    output = model.get_output()

    return output

def run_model_superflexpy_continental(catchments_ids, best_params_dict_model, perm_areas_model):
    # Run the iterative function
    iterative_immediate_downstream = find_iterative_immediate_downstream(df, catchments_ids)

    # Convert results to a DataFrame for display
    iterative_downstream_df = pd.DataFrame(iterative_immediate_downstream.items(), 
                                        columns=['basin_id', 'immediate_downstream_basin'])


    # Assuming the DataFrame has columns 'basin_id' and 'downstream_id'
    topology_list = {basin: None for basin in catchments_ids}  # Default to None

    # Filter DataFrame for relevant basin_ids and update topology
    for _, row in iterative_downstream_df.iterrows():
        if row['basin_id'] in topology_list:
            topology_list[row['basin_id']] = row['immediate_downstream_basin']

    # Generate Nodes dynamically and assign them as global variables
    catchments = [] # Dictionary to store nodes
    
    general = Unit(
    layers=[
        [upper_splitter],
        [snow, upper_transparent],
        [upper_junction],
        [unsaturated],
        [lower_splitter],
        [slow, lag_fun],
        [lower_transparent, fast],
        [lower_junction],
    ],
    id='general')

    low = Unit(
        layers=[
        [upper_splitter],
        [snow, upper_transparent],
        [upper_junction],
        [unsaturated],
        [lower_splitter],
        [slow, lag_fun],
        [lower_transparent, fast],
        [lower_junction],
    ],
        id='low')

    high = Unit(
        layers=[
        [upper_splitter],
        [snow, upper_transparent],
        [upper_junction],
        [unsaturated],
        [lower_splitter],
        [slow, lag_fun],
        [lower_transparent, fast],
        [lower_junction],
    ],
        id='high')

    for cat_id in catchments_ids:
        node = Node(
            units=[high, general, low],  # Use unit from dictionary or default
            weights=perm_areas_model[cat_id],
            area=areas.get(cat_id),  # Use predefined area or default
            id=cat_id
        )
        catchments.append(node)  # Store in the list

        # Assign the node as a global variable
        globals()[cat_id] = node


    # Ensure topology only includes nodes that exist in `catchments_ids`
    topology = {
        cat_id: upstream if upstream in catchments_ids else None
        for cat_id, upstream in topology_list.items() if cat_id in catchments_ids
    }

    # Create the Network
    model = Network(
        nodes=catchments,  # Pass list of Node objects
        topology=topology  
    )

    model.reset_states()

    # Set inputs for each node using the manually defined dictionary
    for cat in catchments:
        cat.set_input(inputs[cat.id])  # Correct way to set inputs

    model.set_timestep(1.0)
    model.set_parameters(best_params_dict_model)
    
    print(model.get_parameters())
    print(model._content[1].get_parameters())
    print(model._content[-1].get_parameters())

    output = model.get_output()

    return output

def run_model_superflexpy_global(catchments_ids, best_params_dict_model, perm_areas_model):
    # Run the iterative function
    iterative_immediate_downstream = find_iterative_immediate_downstream(df, catchments_ids)

    # Convert results to a DataFrame for display
    iterative_downstream_df = pd.DataFrame(iterative_immediate_downstream.items(), 
                                        columns=['basin_id', 'immediate_downstream_basin'])


    # Assuming the DataFrame has columns 'basin_id' and 'downstream_id'
    topology_list = {basin: None for basin in catchments_ids}  # Default to None

    # Filter DataFrame for relevant basin_ids and update topology
    for _, row in iterative_downstream_df.iterrows():
        if row['basin_id'] in topology_list:
            topology_list[row['basin_id']] = row['immediate_downstream_basin']

    # Generate Nodes dynamically and assign them as global variables
    catchments = [] # Dictionary to store nodes
    
    general = Unit(
    layers=[
        [upper_splitter],
        [snow, upper_transparent],
        [upper_junction],
        [unsaturated],
        [lower_splitter],
        [slow, lag_fun],
        [lower_transparent, fast],
        [lower_junction],
    ],
    id='general')

    low = Unit(
        layers=[
        [upper_splitter],
        [snow, upper_transparent],
        [upper_junction],
        [unsaturated],
        [lower_splitter],
        [slow, lag_fun],
        [lower_transparent, fast],
        [lower_junction],
    ],
        id='low')

    high = Unit(
        layers=[
        [upper_splitter],
        [snow, upper_transparent],
        [upper_junction],
        [unsaturated],
        [lower_splitter],
        [slow, lag_fun],
        [lower_transparent, fast],
        [lower_junction],
    ],
        id='high')

    for cat_id in catchments_ids:
        node = Node(
            units=[high, general, low],  # Use unit from dictionary or default
            weights=perm_areas_model[cat_id],
            area=areas.get(cat_id),  # Use predefined area or default
            id=cat_id
        )
        catchments.append(node)  # Store in the list

        # Assign the node as a global variable
        globals()[cat_id] = node


    # Ensure topology only includes nodes that exist in `catchments_ids`
    topology = {
        cat_id: upstream if upstream in catchments_ids else None
        for cat_id, upstream in topology_list.items() if cat_id in catchments_ids
    }

    # Create the Network
    model = Network(
        nodes=catchments,  # Pass list of Node objects
        topology=topology  
    )

    model.reset_states()

    # Set inputs for each node using the manually defined dictionary
    for cat in catchments:
        cat.set_input(inputs[cat.id])  # Correct way to set inputs

    model.set_timestep(1.0)
    model.set_parameters(best_params_dict_model)
    
    print(model.get_parameters())
    print(model._content[1].get_parameters())
    print(model._content[-1].get_parameters())

    output = model.get_output()

    return output


def is_valid_key(k):
    # Exclude keys that end in Group_X_2
    return not re.search(r'Group_\d+_2$', k)

def is_valid_key_2(k):
    # Include only keys that end in Group_X_2
    return re.search(r'Group_\d+_2$', k)


def extract_group_from_key(key):
    """
    Extracts 'Group_X' from keys like:
    - moselle_best_params_regicompt_Group_5
    - moselle_best_params_regicompt_Group_5_2
    """
    match = re.search(r'Group_\d+', key)
    if match is None:
        raise ValueError(f"Could not extract group from key: {key}")
    return match.group(0)


def get_catchments_excluding_group(df, group_to_exclude):
    """
    Returns index values excluding rows belonging to `group_to_exclude`
    """
    return df.loc[df["group"] != group_to_exclude].index.tolist()

## Model all time-series using all possible combinations of params

In [12]:
path_inputs = '../data/models/inputgaronne/subset_2001_2015'

inputs = np.load(path_inputs+'//inputs.npy', allow_pickle=True).item()
observations = np.load(path_inputs+'//observations.npy', allow_pickle=True).item()
areas = np.load(path_inputs+'//areas.npy', allow_pickle=True).item()
perm_areas = np.load(path_inputs+'//perm_areas.npy', allow_pickle=True).item()
perm_areascontinental = np.load(path_inputs+'//perm_areascontinental.npy', allow_pickle=True).item()
perm_areasglobal = np.load(path_inputs+'//perm_areasglobal.npy', allow_pickle=True).item()
quality_masks = np.load(path_inputs+'//quality_masks.npy', allow_pickle=True).item()
rootdepth_mean = np.load(path_inputs+'//rootdepth_mean.npy', allow_pickle=True).item()
waterdeficit_mean= np.load(path_inputs+'//waterdeficit_mean.npy', allow_pickle=True).item()

# Filter keys
regional_keys = [k for k in all_param_dicts if "regi" in k and is_valid_key(k)]
continental_keys = [k for k in all_param_dicts if "cont" in k and is_valid_key(k)]
global_keys = [k for k in all_param_dicts if "glob" in k and is_valid_key(k)]

output_regional_dict = {}
output_continental_dict = {}
output_global_dict = {}

for key in tqdm.tqdm(regional_keys):
    print(f"Running model for key: {key}")
    
    group_to_exclude = extract_group_from_key(key)
    catchments_ids_run = get_catchments_excluding_group(
        estreams_attributes_clipped_filters,
        group_to_exclude
    )

    output = run_model_superflexpy(
        catchments_ids=catchments_ids_run,
        best_params_dict_model=all_param_dicts[key],
        perm_areas_model=perm_areas
    )

    output_regional_dict[key] = output
    print(catchments_ids_run)
    print(len(catchments_ids_run))

    
for key in tqdm.tqdm(continental_keys):
    print(f"Running model for key: {key}")
    group_to_exclude = extract_group_from_key(key)
    catchments_ids_run = get_catchments_excluding_group(
        estreams_attributes_clipped_filters,
        group_to_exclude
    )
    
    output = run_model_superflexpy_continental(
        catchments_ids=catchments_ids_run,
        best_params_dict_model=all_param_dicts[key],
        perm_areas_model=perm_areascontinental
    )

    output_continental_dict[key] = output

for key in tqdm.tqdm(global_keys):
    print(f"Running model for key: {key}")
    group_to_exclude = extract_group_from_key(key)
    catchments_ids_run = get_catchments_excluding_group(
        estreams_attributes_clipped_filters,
        group_to_exclude
    )
    
    output = run_model_superflexpy_global(
        catchments_ids=catchments_ids_run,
        best_params_dict_model=all_param_dicts[key],
        perm_areas_model=perm_areasglobal
    )
    output_global_dict[key] = output

  0%|          | 0/2 [00:00<?, ?it/s]

Running model for key: garonne_best_params_regicompt_Group_1
{'high_snow_t0': 0.2771277, 'high_snow_k': 0.6722174, 'high_snow_m': 2.0, 'high_unsaturated_Smax': 306.55695, 'high_unsaturated_Ce': 0.6848524, 'high_unsaturated_m': 0.01, 'high_unsaturated_beta': 2.6819546, 'high_lowersplitter_splitpar': 0.8999487, 'high_slow_k': 0.0015717044, 'high_slow_alpha': 1.0, 'high_lag-fun_lag-time': 2.000008, 'high_fast_k': 0.7921931, 'high_fast_alpha': 2.0, 'general_snow_t0': 0.2771277, 'general_snow_k': 0.6722174, 'general_snow_m': 2.0, 'general_unsaturated_Smax': 306.55695, 'general_unsaturated_Ce': 0.6848524, 'general_unsaturated_m': 0.01, 'general_unsaturated_beta': 2.6819546, 'general_lowersplitter_splitpar': 0.10002177, 'general_slow_k': 0.09693248, 'general_slow_alpha': 1.0, 'general_lag-fun_lag-time': 2.000008, 'general_fast_k': 0.02696252, 'general_fast_alpha': 2.0, 'low_snow_t0': 0.2771277, 'low_snow_k': 0.6722174, 'low_snow_m': 2.0, 'low_unsaturated_Smax': 306.55695, 'low_unsaturated_Ce'

 50%|█████     | 1/2 [00:12<00:12, 12.25s/it]

['FR004133', 'FR001544', 'FR004137', 'FR001526', 'FR004132', 'FR004114', 'FR004119', 'FR001551', 'FR004162', 'FR004143', 'FR001604']
11
Running model for key: garonne_best_params_regicompt_Group_2
{'high_snow_t0': 0.28083238, 'high_snow_k': 0.31458122, 'high_snow_m': 2.0, 'high_unsaturated_Smax': 195.74603, 'high_unsaturated_Ce': 0.7277381, 'high_unsaturated_m': 0.01, 'high_unsaturated_beta': 2.3687954, 'high_lowersplitter_splitpar': 0.89880043, 'high_slow_k': 0.005143166, 'high_slow_alpha': 1.0, 'high_lag-fun_lag-time': 1.9515992, 'high_fast_k': 0.2670773, 'high_fast_alpha': 2.0, 'general_snow_t0': 0.28083238, 'general_snow_k': 0.31458122, 'general_snow_m': 2.0, 'general_unsaturated_Smax': 195.74603, 'general_unsaturated_Ce': 0.7277381, 'general_unsaturated_m': 0.01, 'general_unsaturated_beta': 2.3687954, 'general_lowersplitter_splitpar': 0.2500758, 'general_slow_k': 0.0004463666, 'general_slow_alpha': 1.0, 'general_lag-fun_lag-time': 1.9515992, 'general_fast_k': 0.03900823, 'general_

100%|██████████| 2/2 [00:23<00:00, 11.69s/it]


...

  0%|          | 0/2 [00:00<?, ?it/s]

Running model for key: garonne_best_params_globcompt_Group_1
{'high_snow_t0': 0.35966292, 'high_snow_k': 0.53716105, 'high_snow_m': 2.0, 'high_unsaturated_Smax': 245.03174, 'high_unsaturated_Ce': 0.71061075, 'high_unsaturated_m': 0.01, 'high_unsaturated_beta': 3.3494465, 'high_lowersplitter_splitpar': 0.8976201, 'high_slow_k': 0.0011783324, 'high_slow_alpha': 1.0, 'high_lag-fun_lag-time': 1.9996614, 'high_fast_k': 0.00040931, 'high_fast_alpha': 2.0, 'general_snow_t0': 0.35966292, 'general_snow_k': 0.53716105, 'general_snow_m': 2.0, 'general_unsaturated_Smax': 245.03174, 'general_unsaturated_Ce': 0.71061075, 'general_unsaturated_m': 0.01, 'general_unsaturated_beta': 3.3494465, 'general_lowersplitter_splitpar': 0.4810594, 'general_slow_k': 0.08148717, 'general_slow_alpha': 1.0, 'general_lag-fun_lag-time': 1.9996614, 'general_fast_k': 0.1225279, 'general_fast_alpha': 2.0, 'low_snow_t0': 0.35966292, 'low_snow_k': 0.53716105, 'low_snow_m': 2.0, 'low_unsaturated_Smax': 245.03174, 'low_unsatu

 50%|█████     | 1/2 [00:08<00:08,  8.93s/it]

Running model for key: garonne_best_params_globcompt_Group_2
{'high_snow_t0': 0.23093516, 'high_snow_k': 0.41207457, 'high_snow_m': 2.0, 'high_unsaturated_Smax': 230.80212, 'high_unsaturated_Ce': 0.70191884, 'high_unsaturated_m': 0.01, 'high_unsaturated_beta': 2.4562309, 'high_lowersplitter_splitpar': 0.8995961, 'high_slow_k': 0.0007673306, 'high_slow_alpha': 1.0, 'high_lag-fun_lag-time': 2.000319, 'high_fast_k': 0.31110194, 'high_fast_alpha': 2.0, 'general_snow_t0': 0.23093516, 'general_snow_k': 0.41207457, 'general_snow_m': 2.0, 'general_unsaturated_Smax': 230.80212, 'general_unsaturated_Ce': 0.70191884, 'general_unsaturated_m': 0.01, 'general_unsaturated_beta': 2.4562309, 'general_lowersplitter_splitpar': 0.39158085, 'general_slow_k': 0.05245534, 'general_slow_alpha': 1.0, 'general_lag-fun_lag-time': 2.000319, 'general_fast_k': 0.054899093, 'general_fast_alpha': 2.0, 'low_snow_t0': 0.23093516, 'low_snow_k': 0.41207457, 'low_snow_m': 2.0, 'low_unsaturated_Smax': 230.80212, 'low_unsat

100%|██████████| 2/2 [00:17<00:00,  8.92s/it]


In [13]:
path_inputs = '../data/models/inputgaronne/subset_1988_2001'

inputs = np.load(path_inputs+'//inputs.npy', allow_pickle=True).item()
observations = np.load(path_inputs+'//observations.npy', allow_pickle=True).item()
areas = np.load(path_inputs+'//areas.npy', allow_pickle=True).item()
perm_areas = np.load(path_inputs+'//perm_areas.npy', allow_pickle=True).item()
perm_areascontinental = np.load(path_inputs+'//perm_areascontinental.npy', allow_pickle=True).item()
perm_areasglobal = np.load(path_inputs+'//perm_areasglobal.npy', allow_pickle=True).item()
quality_masks = np.load(path_inputs+'//quality_masks.npy', allow_pickle=True).item()
rootdepth_mean = np.load(path_inputs+'//rootdepth_mean.npy', allow_pickle=True).item()
waterdeficit_mean= np.load(path_inputs+'//waterdeficit_mean.npy', allow_pickle=True).item()

# Filter keys
regional_keys_2 = [k for k in all_param_dicts if "regi" in k and is_valid_key_2(k)]
continental_keys_2 = [k for k in all_param_dicts if "cont" in k and is_valid_key_2(k)]
global_keys_2 = [k for k in all_param_dicts if "glob" in k and is_valid_key_2(k)]

output_regional_dict_8801 = {}
output_continental_dict_8801 = {}
output_global_dict_8801 = {}

for key in tqdm.tqdm(regional_keys_2):
    print(f"Running model for key: {key}")

    group_to_exclude = extract_group_from_key(key)
    catchments_ids_run = get_catchments_excluding_group(
        estreams_attributes_clipped_filters,
        group_to_exclude
    )
    
    output = run_model_superflexpy(
        catchments_ids=catchments_ids_run,
        best_params_dict_model=all_param_dicts[key],
        perm_areas_model=perm_areas
    )

    output_regional_dict_8801[key] = output

for key in tqdm.tqdm(continental_keys_2):
    print(f"Running model for key: {key}")
    
    group_to_exclude = extract_group_from_key(key)
    catchments_ids_run = get_catchments_excluding_group(
        estreams_attributes_clipped_filters,
        group_to_exclude
    )
    
    output = run_model_superflexpy_continental(
        catchments_ids=catchments_ids_run,
        best_params_dict_model=all_param_dicts[key],
        perm_areas_model=perm_areascontinental
    )

    output_continental_dict_8801[key] = output

for key in tqdm.tqdm(global_keys_2):
    print(f"Running model for key: {key}")
    
    group_to_exclude = extract_group_from_key(key)
    catchments_ids_run = get_catchments_excluding_group(
        estreams_attributes_clipped_filters,
        group_to_exclude
    )
    
    output = run_model_superflexpy_global(
        catchments_ids=catchments_ids_run,
        best_params_dict_model=all_param_dicts[key],
        perm_areas_model=perm_areasglobal
    )
    
    output_global_dict_8801[key] = output

  0%|          | 0/2 [00:00<?, ?it/s]

Running model for key: garonne_best_params_regicompt_Group_2_2
{'high_snow_t0': -0.006356097, 'high_snow_k': 2.255143, 'high_snow_m': 2.0, 'high_unsaturated_Smax': 248.66031, 'high_unsaturated_Ce': 1.0495418, 'high_unsaturated_m': 0.01, 'high_unsaturated_beta': 1.2089936, 'high_lowersplitter_splitpar': 0.898909, 'high_slow_k': 0.0007390456, 'high_slow_alpha': 1.0, 'high_lag-fun_lag-time': 2.0002384, 'high_fast_k': 0.58517295, 'high_fast_alpha': 2.0, 'general_snow_t0': -0.006356097, 'general_snow_k': 2.255143, 'general_snow_m': 2.0, 'general_unsaturated_Smax': 248.66031, 'general_unsaturated_Ce': 1.0495418, 'general_unsaturated_m': 0.01, 'general_unsaturated_beta': 1.2089936, 'general_lowersplitter_splitpar': 0.26941842, 'general_slow_k': 0.040313005, 'general_slow_alpha': 1.0, 'general_lag-fun_lag-time': 2.0002384, 'general_fast_k': 0.04205464, 'general_fast_alpha': 2.0, 'low_snow_t0': -0.006356097, 'low_snow_k': 2.255143, 'low_snow_m': 2.0, 'low_unsaturated_Smax': 248.66031, 'low_unsa

 50%|█████     | 1/2 [00:08<00:08,  8.89s/it]

Running model for key: garonne_best_params_regicompt_Group_1_2
{'high_snow_t0': 0.47387776, 'high_snow_k': 0.3964899, 'high_snow_m': 2.0, 'high_unsaturated_Smax': 228.82904, 'high_unsaturated_Ce': 0.89229304, 'high_unsaturated_m': 0.01, 'high_unsaturated_beta': 2.1522386, 'high_lowersplitter_splitpar': 0.8989098, 'high_slow_k': 0.002499331, 'high_slow_alpha': 1.0, 'high_lag-fun_lag-time': 1.9967147, 'high_fast_k': 0.28968742, 'high_fast_alpha': 2.0, 'general_snow_t0': 0.47387776, 'general_snow_k': 0.3964899, 'general_snow_m': 2.0, 'general_unsaturated_Smax': 228.82904, 'general_unsaturated_Ce': 0.89229304, 'general_unsaturated_m': 0.01, 'general_unsaturated_beta': 2.1522386, 'general_lowersplitter_splitpar': 0.13860163, 'general_slow_k': 0.01617464, 'general_slow_alpha': 1.0, 'general_lag-fun_lag-time': 1.9967147, 'general_fast_k': 0.03440161, 'general_fast_alpha': 2.0, 'low_snow_t0': 0.47387776, 'low_snow_k': 0.3964899, 'low_snow_m': 2.0, 'low_unsaturated_Smax': 228.82904, 'low_unsatu

  0%|          | 0/2 [00:00<?, ?it/s]

...

  0%|          | 0/2 [00:00<?, ?it/s]

Running model for key: garonne_best_params_globcompt_Group_2_2
{'high_snow_t0': 0.007533064, 'high_snow_k': 2.7461567, 'high_snow_m': 2.0, 'high_unsaturated_Smax': 262.0367, 'high_unsaturated_Ce': 1.0321435, 'high_unsaturated_m': 0.01, 'high_unsaturated_beta': 1.2368059, 'high_lowersplitter_splitpar': 0.6909905, 'high_slow_k': 0.0022856453, 'high_slow_alpha': 1.0, 'high_lag-fun_lag-time': 1.9983929, 'high_fast_k': 0.025112003, 'high_fast_alpha': 2.0, 'general_snow_t0': 0.007533064, 'general_snow_k': 2.7461567, 'general_snow_m': 2.0, 'general_unsaturated_Smax': 262.0367, 'general_unsaturated_Ce': 1.0321435, 'general_unsaturated_m': 0.01, 'general_unsaturated_beta': 1.2368059, 'general_lowersplitter_splitpar': 0.39389357, 'general_slow_k': 0.03416761, 'general_slow_alpha': 1.0, 'general_lag-fun_lag-time': 1.9983929, 'general_fast_k': 0.050226845, 'general_fast_alpha': 2.0, 'low_snow_t0': 0.007533064, 'low_snow_k': 2.7461567, 'low_snow_m': 2.0, 'low_unsaturated_Smax': 262.0367, 'low_unsat

 50%|█████     | 1/2 [00:08<00:08,  8.33s/it]

Running model for key: garonne_best_params_globcompt_Group_1_2
{'high_snow_t0': 0.47177216, 'high_snow_k': 0.31452507, 'high_snow_m': 2.0, 'high_unsaturated_Smax': 239.35747, 'high_unsaturated_Ce': 0.8868705, 'high_unsaturated_m': 0.01, 'high_unsaturated_beta': 2.0447364, 'high_lowersplitter_splitpar': 0.8998214, 'high_slow_k': 0.0044627874, 'high_slow_alpha': 1.0, 'high_lag-fun_lag-time': 2.0008867, 'high_fast_k': 0.997817, 'high_fast_alpha': 2.0, 'general_snow_t0': 0.47177216, 'general_snow_k': 0.31452507, 'general_snow_m': 2.0, 'general_unsaturated_Smax': 239.35747, 'general_unsaturated_Ce': 0.8868705, 'general_unsaturated_m': 0.01, 'general_unsaturated_beta': 2.0447364, 'general_lowersplitter_splitpar': 0.41469732, 'general_slow_k': 0.06049598, 'general_slow_alpha': 1.0, 'general_lag-fun_lag-time': 2.0008867, 'general_fast_k': 0.076940484, 'general_fast_alpha': 2.0, 'low_snow_t0': 0.47177216, 'low_snow_k': 0.31452507, 'low_snow_m': 2.0, 'low_unsaturated_Smax': 239.35747, 'low_unsat

100%|██████████| 2/2 [00:16<00:00,  8.40s/it]


In [14]:
# Create the concatenated data for the complete series analysis
path_inputs = '../data/models/inputgaronne/subset_1988_2001'
observations1 = np.load(path_inputs+'//observations.npy', allow_pickle=True).item()
quality_masks1 = np.load(path_inputs+'//quality_masks.npy', allow_pickle=True).item()

path_inputs = '../data/models/inputgaronne/subset_2001_2015'
observations2 = np.load(path_inputs+'//observations.npy', allow_pickle=True).item()
quality_masks2 = np.load(path_inputs+'//quality_masks.npy', allow_pickle=True).item()

observations_cal = {}

for key in observations1.keys():
    arr1 = np.atleast_1d(observations1[key])
    arr2 = np.atleast_1d(observations2.get(key, np.array([])))

    # Always concatenate arrays, even if they contain NaNs or are empty
    #observations_cal[key] = np.concatenate([arr1, arr2])
    # Remove the first 365 days from the second dataset
    arr2_trimmed = arr2[365:] if arr2.size > 365 else np.array([])

    observations_cal[key] = np.concatenate([arr1, arr2_trimmed])

quality_masks_cal = {}

for key in quality_masks1.keys():
    arr1 = np.atleast_1d(quality_masks1[key])
    arr2 = np.atleast_1d(quality_masks2.get(key, np.array([])))

    # Always concatenate arrays, even if they contain NaNs or are empty
    #quality_masks_cal[key] = np.concatenate([arr1, arr2])
    # Remove the first 365 days from the second dataset
    arr2_trimmed = arr2[365:] if arr2.size > 365 else np.array([])

    quality_masks_cal[key] = np.concatenate([arr1, arr2_trimmed])

In [15]:
import numpy as np

# Load both input files
path_inputs_1 = '../data/models/inputgaronne/subset_1988_2001/inputs.npy'
path_inputs_2 = '../data/models/inputgaronne/subset_2001_2015/inputs.npy'

inputs1 = np.load(path_inputs_1, allow_pickle=True).item()
inputs2 = np.load(path_inputs_2, allow_pickle=True).item()

# Initialize new dictionaries
precipitation_cal = {}
temperature_cal = {}
evaporation_cal = {}

for key in inputs1.keys():
    # Get (P, T, PET) tuples from each period
    p1, t1, pet1 = map(np.atleast_1d, inputs1[key])
    p2, t2, pet2 = map(np.atleast_1d, inputs2.get(key, ([], [], [])))


    p2 = p2[365:] if p2.size > 365 else np.array([])
    t2 = t2[365:] if t2.size > 365 else np.array([])
    pet2 = pet2[365:] if pet2.size > 365 else np.array([])


    # Concatenate and assign
    precipitation_cal[key] = np.concatenate([p1, p2])
    temperature_cal[key] = np.concatenate([t1, t2])
    evaporation_cal[key] = np.concatenate([pet1, pet2])


In [16]:
output_global_dict_cal = {}

for param_key in output_global_dict:
    param_key_8801 = param_key + "_2"

    if param_key_8801 in output_global_dict_8801:
        merged_outputs = {}

        for gauge_id in output_global_dict[param_key]:
            if gauge_id in output_global_dict_8801[param_key_8801]:
                # Ensure both arrays are flat (1D)
                series_8801 = np.ravel(output_global_dict_8801[param_key_8801][gauge_id])
                series_recent = np.ravel(output_global_dict[param_key][gauge_id])
                #concatenated = np.concatenate([series_8801, series_recent])
                # Remove first 365 days from recent series
                series_recent_trimmed = series_recent[365:] if series_recent.size > 365 else np.array([])
                concatenated = np.concatenate([series_8801, series_recent_trimmed])
                
                merged_outputs[gauge_id] = [concatenated]

        output_global_dict_cal[param_key] = merged_outputs

In [17]:
output_continental_dict_cal = {}

for param_key in output_continental_dict:
    param_key_8801 = param_key + "_2"

    if param_key_8801 in output_continental_dict_8801:
        merged_outputs = {}

        for gauge_id in output_continental_dict[param_key]:
            if gauge_id in output_continental_dict_8801[param_key_8801]:
                # Ensure both arrays are flat (1D)
                series_8801 = np.ravel(output_continental_dict_8801[param_key_8801][gauge_id])
                series_recent = np.ravel(output_continental_dict[param_key][gauge_id])
                #concatenated = np.concatenate([series_8801, series_recent])
                
                # Remove first 365 days from recent series
                series_recent_trimmed = series_recent[365:] if series_recent.size > 365 else np.array([])
                concatenated = np.concatenate([series_8801, series_recent_trimmed])

                merged_outputs[gauge_id] = [concatenated]

        output_continental_dict_cal[param_key] = merged_outputs

In [18]:
output_regional_dict_cal = {}

for param_key in output_regional_dict:
    param_key_8801 = param_key + "_2"

    if param_key_8801 in output_regional_dict_8801:
        merged_outputs = {}

        for gauge_id in output_regional_dict[param_key]:
            if gauge_id in output_regional_dict_8801[param_key_8801]:
                # Ensure both arrays are flat (1D)
                series_8801 = np.ravel(output_regional_dict_8801[param_key_8801][gauge_id])
                series_recent = np.ravel(output_regional_dict[param_key][gauge_id])
                #concatenated = np.concatenate([series_8801, series_recent])
                
                # Remove first 365 days from recent series
                series_recent_trimmed = series_recent[365:] if series_recent.size > 365 else np.array([])
                concatenated = np.concatenate([series_8801, series_recent_trimmed])

                merged_outputs[gauge_id] = [concatenated]

        output_regional_dict_cal[param_key] = merged_outputs

## Space-time validation

In [19]:
path_inputs = '../data/models/inputgaronne/subset_1988_2001'

inputs = np.load(path_inputs+'//inputs.npy', allow_pickle=True).item()
observations = np.load(path_inputs+'//observations.npy', allow_pickle=True).item()
areas = np.load(path_inputs+'//areas.npy', allow_pickle=True).item()
perm_areas = np.load(path_inputs+'//perm_areas.npy', allow_pickle=True).item()
perm_areascontinental = np.load(path_inputs+'//perm_areascontinental.npy', allow_pickle=True).item()
perm_areasglobal = np.load(path_inputs+'//perm_areasglobal.npy', allow_pickle=True).item()
quality_masks = np.load(path_inputs+'//quality_masks.npy', allow_pickle=True).item()
rootdepth_mean = np.load(path_inputs+'//rootdepth_mean.npy', allow_pickle=True).item()
waterdeficit_mean= np.load(path_inputs+'//waterdeficit_mean.npy', allow_pickle=True).item()

# Filter keys
regional_keys = [k for k in all_param_dicts if "regi" in k and is_valid_key(k)]
continental_keys = [k for k in all_param_dicts if "cont" in k and is_valid_key(k)]
global_keys = [k for k in all_param_dicts if "glob" in k and is_valid_key(k)]

output_regional_val_dict = {}
output_continental_val_dict = {}
output_global_val_dict = {}

for key in tqdm.tqdm(regional_keys):
    print(f"Running model for key: {key}")
    
    group_to_exclude = extract_group_from_key(key)
    catchments_ids_run = get_catchments_excluding_group(
        estreams_attributes_clipped_filters,
        group_to_exclude
    )
    
    output = run_model_superflexpy(
        catchments_ids=catchments_ids_run,
        best_params_dict_model=all_param_dicts[key],
        perm_areas_model=perm_areas
    )

    output_regional_val_dict[key] = output

for key in tqdm.tqdm(continental_keys):
    print(f"Running model for key: {key}")
    
    group_to_exclude = extract_group_from_key(key)
    catchments_ids_run = get_catchments_excluding_group(
        estreams_attributes_clipped_filters,
        group_to_exclude
    )
    
    output = run_model_superflexpy_continental(
        catchments_ids=catchments_ids_run,
        best_params_dict_model=all_param_dicts[key],
        perm_areas_model=perm_areascontinental
    )

    output_continental_val_dict[key] = output

for key in tqdm.tqdm(global_keys):
    print(f"Running model for key: {key}")

    group_to_exclude = extract_group_from_key(key)
    catchments_ids_run = get_catchments_excluding_group(
        estreams_attributes_clipped_filters,
        group_to_exclude
    )
    
    output = run_model_superflexpy_global(
        catchments_ids=catchments_ids_run,
        best_params_dict_model=all_param_dicts[key],
        perm_areas_model=perm_areasglobal
    )

    output_global_val_dict[key] = output

  0%|          | 0/2 [00:00<?, ?it/s]

Running model for key: garonne_best_params_regicompt_Group_1
{'high_snow_t0': 0.2771277, 'high_snow_k': 0.6722174, 'high_snow_m': 2.0, 'high_unsaturated_Smax': 306.55695, 'high_unsaturated_Ce': 0.6848524, 'high_unsaturated_m': 0.01, 'high_unsaturated_beta': 2.6819546, 'high_lowersplitter_splitpar': 0.8999487, 'high_slow_k': 0.0015717044, 'high_slow_alpha': 1.0, 'high_lag-fun_lag-time': 2.000008, 'high_fast_k': 0.7921931, 'high_fast_alpha': 2.0, 'general_snow_t0': 0.2771277, 'general_snow_k': 0.6722174, 'general_snow_m': 2.0, 'general_unsaturated_Smax': 306.55695, 'general_unsaturated_Ce': 0.6848524, 'general_unsaturated_m': 0.01, 'general_unsaturated_beta': 2.6819546, 'general_lowersplitter_splitpar': 0.10002177, 'general_slow_k': 0.09693248, 'general_slow_alpha': 1.0, 'general_lag-fun_lag-time': 2.000008, 'general_fast_k': 0.02696252, 'general_fast_alpha': 2.0, 'low_snow_t0': 0.2771277, 'low_snow_k': 0.6722174, 'low_snow_m': 2.0, 'low_unsaturated_Smax': 306.55695, 'low_unsaturated_Ce'

 50%|█████     | 1/2 [00:09<00:09,  9.78s/it]

Running model for key: garonne_best_params_regicompt_Group_2
{'high_snow_t0': 0.28083238, 'high_snow_k': 0.31458122, 'high_snow_m': 2.0, 'high_unsaturated_Smax': 195.74603, 'high_unsaturated_Ce': 0.7277381, 'high_unsaturated_m': 0.01, 'high_unsaturated_beta': 2.3687954, 'high_lowersplitter_splitpar': 0.89880043, 'high_slow_k': 0.005143166, 'high_slow_alpha': 1.0, 'high_lag-fun_lag-time': 1.9515992, 'high_fast_k': 0.2670773, 'high_fast_alpha': 2.0, 'general_snow_t0': 0.28083238, 'general_snow_k': 0.31458122, 'general_snow_m': 2.0, 'general_unsaturated_Smax': 195.74603, 'general_unsaturated_Ce': 0.7277381, 'general_unsaturated_m': 0.01, 'general_unsaturated_beta': 2.3687954, 'general_lowersplitter_splitpar': 0.2500758, 'general_slow_k': 0.0004463666, 'general_slow_alpha': 1.0, 'general_lag-fun_lag-time': 1.9515992, 'general_fast_k': 0.03900823, 'general_fast_alpha': 2.0, 'low_snow_t0': 0.28083238, 'low_snow_k': 0.31458122, 'low_snow_m': 2.0, 'low_unsaturated_Smax': 195.74603, 'low_unsatu

  0%|          | 0/2 [00:00<?, ?it/s]

...

  0%|          | 0/2 [00:00<?, ?it/s]

Running model for key: garonne_best_params_globcompt_Group_1
{'high_snow_t0': 0.35966292, 'high_snow_k': 0.53716105, 'high_snow_m': 2.0, 'high_unsaturated_Smax': 245.03174, 'high_unsaturated_Ce': 0.71061075, 'high_unsaturated_m': 0.01, 'high_unsaturated_beta': 3.3494465, 'high_lowersplitter_splitpar': 0.8976201, 'high_slow_k': 0.0011783324, 'high_slow_alpha': 1.0, 'high_lag-fun_lag-time': 1.9996614, 'high_fast_k': 0.00040931, 'high_fast_alpha': 2.0, 'general_snow_t0': 0.35966292, 'general_snow_k': 0.53716105, 'general_snow_m': 2.0, 'general_unsaturated_Smax': 245.03174, 'general_unsaturated_Ce': 0.71061075, 'general_unsaturated_m': 0.01, 'general_unsaturated_beta': 3.3494465, 'general_lowersplitter_splitpar': 0.4810594, 'general_slow_k': 0.08148717, 'general_slow_alpha': 1.0, 'general_lag-fun_lag-time': 1.9996614, 'general_fast_k': 0.1225279, 'general_fast_alpha': 2.0, 'low_snow_t0': 0.35966292, 'low_snow_k': 0.53716105, 'low_snow_m': 2.0, 'low_unsaturated_Smax': 245.03174, 'low_unsatu

 50%|█████     | 1/2 [00:09<00:09,  9.69s/it]

Running model for key: garonne_best_params_globcompt_Group_2
{'high_snow_t0': 0.23093516, 'high_snow_k': 0.41207457, 'high_snow_m': 2.0, 'high_unsaturated_Smax': 230.80212, 'high_unsaturated_Ce': 0.70191884, 'high_unsaturated_m': 0.01, 'high_unsaturated_beta': 2.4562309, 'high_lowersplitter_splitpar': 0.8995961, 'high_slow_k': 0.0007673306, 'high_slow_alpha': 1.0, 'high_lag-fun_lag-time': 2.000319, 'high_fast_k': 0.31110194, 'high_fast_alpha': 2.0, 'general_snow_t0': 0.23093516, 'general_snow_k': 0.41207457, 'general_snow_m': 2.0, 'general_unsaturated_Smax': 230.80212, 'general_unsaturated_Ce': 0.70191884, 'general_unsaturated_m': 0.01, 'general_unsaturated_beta': 2.4562309, 'general_lowersplitter_splitpar': 0.39158085, 'general_slow_k': 0.05245534, 'general_slow_alpha': 1.0, 'general_lag-fun_lag-time': 2.000319, 'general_fast_k': 0.054899093, 'general_fast_alpha': 2.0, 'low_snow_t0': 0.23093516, 'low_snow_k': 0.41207457, 'low_snow_m': 2.0, 'low_unsaturated_Smax': 230.80212, 'low_unsat

100%|██████████| 2/2 [00:19<00:00,  9.63s/it]


In [20]:
path_inputs = '../data/models/inputgaronne/subset_2001_2015'

inputs = np.load(path_inputs+'//inputs.npy', allow_pickle=True).item()
observations = np.load(path_inputs+'//observations.npy', allow_pickle=True).item()
areas = np.load(path_inputs+'//areas.npy', allow_pickle=True).item()
perm_areas = np.load(path_inputs+'//perm_areas.npy', allow_pickle=True).item()
perm_areascontinental = np.load(path_inputs+'//perm_areascontinental.npy', allow_pickle=True).item()
perm_areasglobal = np.load(path_inputs+'//perm_areasglobal.npy', allow_pickle=True).item()
quality_masks = np.load(path_inputs+'//quality_masks.npy', allow_pickle=True).item()
rootdepth_mean = np.load(path_inputs+'//rootdepth_mean.npy', allow_pickle=True).item()
waterdeficit_mean= np.load(path_inputs+'//waterdeficit_mean.npy', allow_pickle=True).item()

# Filter keys
regional_keys_2 = [k for k in all_param_dicts if "regi" in k and is_valid_key_2(k)]
continental_keys_2 = [k for k in all_param_dicts if "cont" in k and is_valid_key_2(k)]
global_keys_2 = [k for k in all_param_dicts if "glob" in k and is_valid_key_2(k)]

output_regional_dict_0115 = {}
output_continental_dict_0115 = {}
output_global_dict_0115 = {}

for key in tqdm.tqdm(regional_keys_2):
    print(f"Running model for key: {key}")
    
    group_to_exclude = extract_group_from_key(key)
    catchments_ids_run = get_catchments_excluding_group(
        estreams_attributes_clipped_filters,
        group_to_exclude
    )
    
    output = run_model_superflexpy(
        catchments_ids=catchments_ids_run,
        best_params_dict_model=all_param_dicts[key],
        perm_areas_model=perm_areas
    )

    output_regional_dict_0115[key] = output

for key in tqdm.tqdm(continental_keys_2):
    print(f"Running model for key: {key}")
    
    group_to_exclude = extract_group_from_key(key)
    catchments_ids_run = get_catchments_excluding_group(
        estreams_attributes_clipped_filters,
        group_to_exclude
    )
    
    output = run_model_superflexpy_continental(
        catchments_ids=catchments_ids_run,
        best_params_dict_model=all_param_dicts[key],
        perm_areas_model=perm_areascontinental
    )
    output_continental_dict_0115[key] = output

for key in tqdm.tqdm(global_keys_2):
    print(f"Running model for key: {key}")
    
    group_to_exclude = extract_group_from_key(key)
    catchments_ids_run = get_catchments_excluding_group(
        estreams_attributes_clipped_filters,
        group_to_exclude
    )
    
    output = run_model_superflexpy_global(
        catchments_ids=catchments_ids_run,
        best_params_dict_model=all_param_dicts[key],
        perm_areas_model=perm_areasglobal
    )
    output_global_dict_0115[key] = output

  0%|          | 0/2 [00:00<?, ?it/s]

Running model for key: garonne_best_params_regicompt_Group_2_2
{'high_snow_t0': -0.006356097, 'high_snow_k': 2.255143, 'high_snow_m': 2.0, 'high_unsaturated_Smax': 248.66031, 'high_unsaturated_Ce': 1.0495418, 'high_unsaturated_m': 0.01, 'high_unsaturated_beta': 1.2089936, 'high_lowersplitter_splitpar': 0.898909, 'high_slow_k': 0.0007390456, 'high_slow_alpha': 1.0, 'high_lag-fun_lag-time': 2.0002384, 'high_fast_k': 0.58517295, 'high_fast_alpha': 2.0, 'general_snow_t0': -0.006356097, 'general_snow_k': 2.255143, 'general_snow_m': 2.0, 'general_unsaturated_Smax': 248.66031, 'general_unsaturated_Ce': 1.0495418, 'general_unsaturated_m': 0.01, 'general_unsaturated_beta': 1.2089936, 'general_lowersplitter_splitpar': 0.26941842, 'general_slow_k': 0.040313005, 'general_slow_alpha': 1.0, 'general_lag-fun_lag-time': 2.0002384, 'general_fast_k': 0.04205464, 'general_fast_alpha': 2.0, 'low_snow_t0': -0.006356097, 'low_snow_k': 2.255143, 'low_snow_m': 2.0, 'low_unsaturated_Smax': 248.66031, 'low_unsa

 50%|█████     | 1/2 [00:09<00:09,  9.16s/it]

Running model for key: garonne_best_params_regicompt_Group_1_2
{'high_snow_t0': 0.47387776, 'high_snow_k': 0.3964899, 'high_snow_m': 2.0, 'high_unsaturated_Smax': 228.82904, 'high_unsaturated_Ce': 0.89229304, 'high_unsaturated_m': 0.01, 'high_unsaturated_beta': 2.1522386, 'high_lowersplitter_splitpar': 0.8989098, 'high_slow_k': 0.002499331, 'high_slow_alpha': 1.0, 'high_lag-fun_lag-time': 1.9967147, 'high_fast_k': 0.28968742, 'high_fast_alpha': 2.0, 'general_snow_t0': 0.47387776, 'general_snow_k': 0.3964899, 'general_snow_m': 2.0, 'general_unsaturated_Smax': 228.82904, 'general_unsaturated_Ce': 0.89229304, 'general_unsaturated_m': 0.01, 'general_unsaturated_beta': 2.1522386, 'general_lowersplitter_splitpar': 0.13860163, 'general_slow_k': 0.01617464, 'general_slow_alpha': 1.0, 'general_lag-fun_lag-time': 1.9967147, 'general_fast_k': 0.03440161, 'general_fast_alpha': 2.0, 'low_snow_t0': 0.47387776, 'low_snow_k': 0.3964899, 'low_snow_m': 2.0, 'low_unsaturated_Smax': 228.82904, 'low_unsatu

  0%|          | 0/2 [00:00<?, ?it/s]

...

  0%|          | 0/2 [00:00<?, ?it/s]

Running model for key: garonne_best_params_globcompt_Group_2_2
{'high_snow_t0': 0.007533064, 'high_snow_k': 2.7461567, 'high_snow_m': 2.0, 'high_unsaturated_Smax': 262.0367, 'high_unsaturated_Ce': 1.0321435, 'high_unsaturated_m': 0.01, 'high_unsaturated_beta': 1.2368059, 'high_lowersplitter_splitpar': 0.6909905, 'high_slow_k': 0.0022856453, 'high_slow_alpha': 1.0, 'high_lag-fun_lag-time': 1.9983929, 'high_fast_k': 0.025112003, 'high_fast_alpha': 2.0, 'general_snow_t0': 0.007533064, 'general_snow_k': 2.7461567, 'general_snow_m': 2.0, 'general_unsaturated_Smax': 262.0367, 'general_unsaturated_Ce': 1.0321435, 'general_unsaturated_m': 0.01, 'general_unsaturated_beta': 1.2368059, 'general_lowersplitter_splitpar': 0.39389357, 'general_slow_k': 0.03416761, 'general_slow_alpha': 1.0, 'general_lag-fun_lag-time': 1.9983929, 'general_fast_k': 0.050226845, 'general_fast_alpha': 2.0, 'low_snow_t0': 0.007533064, 'low_snow_k': 2.7461567, 'low_snow_m': 2.0, 'low_unsaturated_Smax': 262.0367, 'low_unsat

 50%|█████     | 1/2 [00:08<00:08,  8.68s/it]

Running model for key: garonne_best_params_globcompt_Group_1_2
{'high_snow_t0': 0.47177216, 'high_snow_k': 0.31452507, 'high_snow_m': 2.0, 'high_unsaturated_Smax': 239.35747, 'high_unsaturated_Ce': 0.8868705, 'high_unsaturated_m': 0.01, 'high_unsaturated_beta': 2.0447364, 'high_lowersplitter_splitpar': 0.8998214, 'high_slow_k': 0.0044627874, 'high_slow_alpha': 1.0, 'high_lag-fun_lag-time': 2.0008867, 'high_fast_k': 0.997817, 'high_fast_alpha': 2.0, 'general_snow_t0': 0.47177216, 'general_snow_k': 0.31452507, 'general_snow_m': 2.0, 'general_unsaturated_Smax': 239.35747, 'general_unsaturated_Ce': 0.8868705, 'general_unsaturated_m': 0.01, 'general_unsaturated_beta': 2.0447364, 'general_lowersplitter_splitpar': 0.41469732, 'general_slow_k': 0.06049598, 'general_slow_alpha': 1.0, 'general_lag-fun_lag-time': 2.0008867, 'general_fast_k': 0.076940484, 'general_fast_alpha': 2.0, 'low_snow_t0': 0.47177216, 'low_snow_k': 0.31452507, 'low_snow_m': 2.0, 'low_unsaturated_Smax': 239.35747, 'low_unsat

100%|██████████| 2/2 [00:17<00:00,  8.66s/it]


In [21]:
output_global_dict_val = {}

for param_key in output_global_val_dict:
    param_key_8801 = param_key + "_2"

    if param_key_8801 in output_global_dict_0115:
        merged_outputs = {}

        for gauge_id in output_global_val_dict[param_key]:
            if gauge_id in output_global_dict_0115[param_key_8801]:
                # Ensure both arrays are flat (1D)
                series_8801 = np.ravel(output_global_dict_0115[param_key_8801][gauge_id])
                series_recent = np.ravel(output_global_val_dict[param_key][gauge_id])
                #concatenated = np.concatenate([series_recent, series_8801])
                
                # Remove first 365 days from recent series
                series_8801_trimmed = series_8801[365:] if series_8801.size > 365 else np.array([])
                concatenated = np.concatenate([series_recent, series_8801_trimmed])

                
                merged_outputs[gauge_id] = [concatenated]

        output_global_dict_val[param_key] = merged_outputs

In [22]:
output_continental_dict_val = {}

for param_key in output_continental_val_dict:
    param_key_8801 = param_key + "_2"

    if param_key_8801 in output_continental_dict_0115:
        merged_outputs = {}

        for gauge_id in output_continental_val_dict[param_key]:
            if gauge_id in output_continental_dict_0115[param_key_8801]:
                # Ensure both arrays are flat (1D)
                series_8801 = np.ravel(output_continental_dict_0115[param_key_8801][gauge_id])
                series_recent = np.ravel(output_continental_val_dict[param_key][gauge_id])
                #concatenated = np.concatenate([series_recent, series_8801])
                
                # Remove first 365 days from recent series
                series_8801_trimmed = series_8801[365:] if series_8801.size > 365 else np.array([])
                concatenated = np.concatenate([series_recent, series_8801_trimmed])

                merged_outputs[gauge_id] = [concatenated]

        output_continental_dict_val[param_key] = merged_outputs

In [23]:
output_regional_dict_val = {}

for param_key in output_regional_val_dict:
    param_key_8801 = param_key + "_2"

    if param_key_8801 in output_regional_dict_0115:
        merged_outputs = {}

        for gauge_id in output_regional_val_dict[param_key]:
            if gauge_id in output_regional_dict_0115[param_key_8801]:
                # Ensure both arrays are flat (1D)
                series_8801 = np.ravel(output_regional_dict_0115[param_key_8801][gauge_id])
                series_recent = np.ravel(output_regional_val_dict[param_key][gauge_id])
                #concatenated = np.concatenate([series_recent, series_8801])
                
                # Remove first 365 days from recent series
                series_8801_trimmed = series_8801[365:] if series_8801.size > 365 else np.array([])
                concatenated = np.concatenate([series_recent, series_8801_trimmed])

                merged_outputs[gauge_id] = [concatenated]

        output_regional_dict_val[param_key] = merged_outputs

## Save the time-series in netcdfs

In [24]:
import xarray as xr
import numpy as np

# Adjust these according to your data
group_suffixes = ["Group_1", 
                  "Group_2"
                  ]

gauge_ids = list(observations_cal.keys())  # Your observations dict should be preloaded
time_index = pd.date_range(start="1988-10-01", end="2015-09-30", freq='D')

In [25]:
ds_inputs = xr.Dataset(
    data_vars={
        "observation": (["gauge_id", "date"], [observations_cal[g] for g in gauge_ids]),
        "precipitation": (["gauge_id", "date"], [precipitation_cal[g] for g in gauge_ids]),
        "temperature": (["gauge_id", "date"], [temperature_cal[g] for g in gauge_ids]),
        "evaporation": (["gauge_id", "date"], [evaporation_cal[g] for g in gauge_ids]),
    },
    coords={
        "gauge_id": gauge_ids,
        "date": time_index
    }
)

ds_inputs.to_netcdf(rf"../results/sim_garonne/space-time/notconcatenated/inputs.nc", engine="scipy")

In [26]:
ds_inputs = xr.Dataset(
    data_vars={
        "observation": (["gauge_id", "date"], [observations_cal[g] for g in gauge_ids]),
        "precipitation": (["gauge_id", "date"], [precipitation_cal[g] for g in gauge_ids]),
        "temperature": (["gauge_id", "date"], [temperature_cal[g] for g in gauge_ids]),
        "evaporation": (["gauge_id", "date"], [evaporation_cal[g] for g in gauge_ids]),
    },
    coords={
        "gauge_id": gauge_ids,
        "date": time_index
    }
)

ds_inputs.to_netcdf(rf"../results/sim_garonne/space/notconcatenated/inputs.nc", engine="scipy")

In [27]:
# Build datasets
datasets = {}

for suffix in group_suffixes:
    reg_key = f"garonne_best_params_regicompt_{suffix}"
    cont_key = f"garonne_best_params_contcompt_{suffix}"
    glob_key = f"garonne_best_params_globcompt_{suffix}"

    reg_data = []
    cont_data = []
    glob_data = []
    group_gauge_ids = []

    for gauge in gauge_ids:
        if gauge in output_regional_dict_val[reg_key] and \
           gauge in output_continental_dict_val[cont_key] and \
           gauge in output_global_dict_val[glob_key]:

            reg_data.append(output_regional_dict_val[reg_key][gauge][0])
            cont_data.append(output_continental_dict_val[cont_key][gauge][0])
            glob_data.append(output_global_dict_val[glob_key][gauge][0])
            group_gauge_ids.append(gauge)

    if group_gauge_ids:
        ds = xr.Dataset(
            data_vars={
                "regional": (["gauge_id", "date"], reg_data),
                "continental": (["gauge_id", "date"], cont_data),
                "global": (["gauge_id", "date"], glob_data)
            },
            coords={
                "gauge_id": group_gauge_ids,
                "date": time_index
            }
        )
        datasets[suffix] = ds

# Save each group to a separate NetCDF file using scipy (no need for netCDF4)
for suffix, ds in datasets.items():
    ds.to_netcdf(rf"../results/sim_garonne/space-time/notconcatenated/simu_compl_{suffix}.nc", engine="scipy")


In [28]:
# Build datasets
datasets = {}

for suffix in group_suffixes:
    reg_key = f"garonne_best_params_regicompt_{suffix}"
    cont_key = f"garonne_best_params_contcompt_{suffix}"
    glob_key = f"garonne_best_params_globcompt_{suffix}"

    reg_data = []
    cont_data = []
    glob_data = []
    group_gauge_ids = []

    for gauge in gauge_ids:
        if gauge in output_regional_dict_cal[reg_key] and \
           gauge in output_continental_dict_cal[cont_key] and \
           gauge in output_global_dict_cal[glob_key]:

            reg_data.append(output_regional_dict_cal[reg_key][gauge][0])
            cont_data.append(output_continental_dict_cal[cont_key][gauge][0])
            glob_data.append(output_global_dict_cal[glob_key][gauge][0])
            group_gauge_ids.append(gauge)

    if group_gauge_ids:
        ds = xr.Dataset(
            data_vars={
                "regional": (["gauge_id", "date"], reg_data),
                "continental": (["gauge_id", "date"], cont_data),
                "global": (["gauge_id", "date"], glob_data)
            },
            coords={
                "gauge_id": group_gauge_ids,
                "date": time_index
            }
        )
        datasets[suffix] = ds

# Save each group to a separate NetCDF file using scipy (no need for netCDF4)
for suffix, ds in datasets.items():
    ds.to_netcdf(rf"../results/sim_garonne/space/notconcatenated/simu_compl_{suffix}.nc", engine="scipy")


# End